# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

***Finding A — Random Forest feature importance for Health Score (page 27)***

**What it claims**: Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the strongest predictors of Health Score — together with CTR (8%), they account for ~98% of the model's importance.

**Where the label comes from**: Health Score isn't an independently measured outcome — it's a hand-built composite (page 5): Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts). The Random Forest is then trained to predict this composite using, among other things, those same four raw inputs.

**My methodology question**: Since the target is a deterministic combination of the top four "predictive" features, doesn't the importance ranking mostly describe the scoring formula rather than revealing what independently drives real search outcomes? The paper calls this relationship "partly" circular — I'd ask what the same Random Forest would show if retrained on an outcome that isn't built from these inputs, like next-month clicks or session growth. That would separate "the model can reconstruct its own recipe" from "the model found something new.


***Finding B — Growth-prediction logistic regression, 71% holdout accuracy***

**What it claims**: A logistic regression predicting growing-vs-declining pages reaches 71% accuracy on an 80/20 holdout split.

**Where the label comes from**: The growth/decline label comes from the same 61.8K-page active-content sample used throughout the ML appendix.

**My methodology question**: The paper doesn't say how the split was made — row-level or brand-grouped — or report the base rate, or confirm all 57 brands appear on both sides. A row-level split lets the model see other pages from the same brand during training, so 71% could partly reflect "this smells like Brand X" rather than a general growth pattern. I ran this exact comparison on my own Week 5 model: switching from a random split to a client-grouped split dropped Precision@10 from 0.82 to 0.64 — a large gap from changing nothing but the split method. Without knowing which split the paper used, 71% is hard to interpret as a real signal versus an inflated one.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Re-used code:** The data loading, rule rebuild (`build_rule_df`), labeling (`build_labelable`), feature construction (`build_X`), and evaluation helper (`precision_at_k`) in this section are carried over from `work/notebooks/w05_model.ipynb`, refactored into standalone functions so they can run against multiple decision points (feb, march) without duplicating logic. The modeling choices — Random Forest, same 8 features, same eligibility filter (≥50 GSC impressions OR ≥10 GA4 sessions), same decline definition (April/March clicks < 80% of prior month, min 5 prior-month clicks) — are unchanged from Week 5. What's new in this notebook is the split comparison itself: the random-vs-grouped test and the added time-aware split.

**Results:**

- BEFORE (random split): 0.90
- AFTER (client-grouped): 0.68
- AFTER (time-aware, feb→mar trained, tested on mar→apr): 0.70

**Before/after read:** Precision@10 drops from 0.90 under a random split to 0.68 under a client-grouped split and 0.70 under a time-aware split (trained on a Feb→Mar decision point, tested on the existing Mar→Apr population). Both honest splits land close to each other despite testing different things — one holds out unseen *clients*, the other holds out an unseen *time period* — which is a stronger signal than either alone: it suggests the random split's 0.90 was inflated by two separate easy-exam effects (memorizing clients, and implicitly reusing the training time window), and the true generalization performance is closer to ~0.68–0.70. That's a meaningful drop from the random number, but not a collapse to chance — the model retains real signal, just less than the random split made it look like.

**Caveat:** the time-aware test still evaluates on largely the same set of pages, just a different month's snapshot of them — it proves the model isn't purely overfit to one calendar window, but it doesn't by itself prove generalization to brand-new content the way the grouped split does. Those two tests are complementary, not redundant.

In [1]:
#load previous data
!pip install -q duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

def load_month_agg(month_str):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_sum_position) AS gsc_sum_position,
               SUM(ga4_sessions) AS ga4_sessions,
               SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM read_parquet('{base}/fact_content_daily_performance/month={month_str}/data_0.parquet')
        GROUP BY client_hash_id, content_hash_id
    """).df()

df_feb_agg = load_month_agg('2026-02')
df_march_agg = load_month_agg('2026-03')
df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feb: {len(df_feb_agg)}, March: {len(df_march_agg)}, April(clicks only): {len(df_april_agg)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb: 321546, March: 331437, April(clicks only): 362172


In [2]:
def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

In [3]:
def build_rule_df(raw_agg):
    d = raw_agg.copy()
    d['gsc_avg_position'] = d['gsc_sum_position'] / d['gsc_impressions']
    d.loc[d['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

    signal = d[d['gsc_impressions'] > 0].copy()
    signal['ctr'] = signal['gsc_clicks'] / signal['gsc_impressions']
    signal['position_bucket'] = signal['gsc_avg_position'].apply(position_bucket)
    peer_ctr = signal.groupby('position_bucket')['ctr'].mean().to_dict()

    d['ctr'] = d['gsc_clicks'] / d['gsc_impressions']
    d['engagement_rate'] = d['ga4_engaged_sessions'] / d['ga4_sessions']
    d['position_bucket'] = d['gsc_avg_position'].apply(position_bucket)
    d['peer_avg_ctr'] = d['position_bucket'].map(peer_ctr)

    eligible = (d['gsc_impressions'] >= 50) | (d['ga4_sessions'] >= 10)
    return d[eligible].copy()

march_rule_df = build_rule_df(df_march_agg)
feb_rule_df = build_rule_df(df_feb_agg)
print(f"March-eligible: {len(march_rule_df)}, Feb-eligible: {len(feb_rule_df)}")

March-eligible: 116512, Feb-eligible: 93871


In [4]:
MIN_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

def build_labelable(rule_df, next_month_clicks_df, next_month_col):
    merged = rule_df.merge(next_month_clicks_df, on=['client_hash_id', 'content_hash_id'], how='left')
    tracked = merged[next_month_col].notna()
    lab = merged[tracked & (merged['gsc_clicks'] >= MIN_CLICKS_FOR_LABEL)].copy()
    lab['decline_label'] = (lab[next_month_col] < DECLINE_THRESHOLD * lab['gsc_clicks']).astype(int)
    return lab

labelable = build_labelable(march_rule_df, df_april_agg, 'gsc_clicks_apr')  # march -> april (existing)

mar_clicks_only = df_march_agg[['client_hash_id', 'content_hash_id', 'gsc_clicks']].rename(
    columns={'gsc_clicks': 'gsc_clicks_mar'})
feb_labelable = build_labelable(feb_rule_df, mar_clicks_only, 'gsc_clicks_mar')  # feb -> march (new)

print(f"March->April labelable: {len(labelable)}, base rate {labelable['decline_label'].mean():.3f}")
print(f"Feb->March labelable:   {len(feb_labelable)}, base rate {feb_labelable['decline_label'].mean():.3f}")

March->April labelable: 28805, base rate 0.545
Feb->March labelable:   21773, base rate 0.313


In [5]:
FEATURES = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
            'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']

def build_X(df, feature_cols):
    X = df[feature_cols].copy()
    worst = X['gsc_avg_position'].max()
    X['gsc_avg_position'] = X['gsc_avg_position'].fillna(worst + 10 if pd.notna(worst) else 100)
    X['ga4_sessions'] = X['ga4_sessions'].fillna(0)
    X['ga4_engaged_sessions'] = X['ga4_engaged_sessions'].fillna(0)
    X['engagement_rate'] = X['engagement_rate'].fillna(0)
    X['ctr'] = X['ctr'].fillna(0)
    X['peer_avg_ctr'] = X['peer_avg_ctr'].fillna(X['peer_avg_ctr'].median())
    X = X.join(pd.get_dummies(df['position_bucket'], prefix='pos', drop_first=True))
    return X

def precision_at_k(ids, scores, labels, k):
    temp = pd.DataFrame({'content_hash_id': ids, 'score': scores, 'label': labels})
    ranked = temp.sort_values(['score', 'content_hash_id'], ascending=[False, True])
    return ranked.head(k)['label'].mean()

X_full = build_X(labelable, FEATURES)
y = labelable['decline_label'].values
groups = labelable['client_hash_id'].values
print(f"X_full: {X_full.shape}, base rate {y.mean():.3f}")

X_full: (28805, 11), base rate 0.545


In [6]:
#group split
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)

kf = KFold(n_splits=5, shuffle=True, random_state=0)
random_p10 = []
for train, test in kf.split(X_full):
    rf_model.fit(X_full.iloc[train], y[train])
    preds = rf_model.predict_proba(X_full.iloc[test])[:, 1]
    random_p10.append(precision_at_k(labelable['content_hash_id'].values[test], preds, y[test], 10))

gkf = GroupKFold(n_splits=5)
grouped_p10 = []
for train, test in gkf.split(X_full, y, groups):
    rf_model.fit(X_full.iloc[train], y[train])
    preds = rf_model.predict_proba(X_full.iloc[test])[:, 1]
    grouped_p10.append(precision_at_k(labelable['content_hash_id'].values[test], preds, y[test], 10))

print(f"BEFORE (random split):  {np.mean(random_p10):.2f}")
print(f"AFTER (grouped split):  {np.mean(grouped_p10):.2f}")

BEFORE (random split):  0.90
AFTER (grouped split):  0.58


In [7]:
# time aware split
X_feb = build_X(feb_labelable, FEATURES)
y_feb = feb_labelable['decline_label'].values

rf_time_aware = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)
rf_time_aware.fit(X_feb, y_feb)

time_preds = rf_time_aware.predict_proba(X_full)[:, 1]  # X_full/y/labelable = march-decision test set
time_p10 = precision_at_k(labelable['content_hash_id'].values, time_preds, y, 10)

print(f"BEFORE (random split):  {np.mean(random_p10):.2f}")
print(f"AFTER (client-grouped): {np.mean(grouped_p10):.2f}")
print(f"AFTER (time-aware, feb-mar trained, tested on mar-apr):  {time_p10:.2f}")

BEFORE (random split):  0.90
AFTER (client-grouped): 0.58
AFTER (time-aware, feb-mar trained, tested on mar-apr):  0.60


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Three ways an answer can sneak in, checked against `FEATURES` and `X_full` as actually used in Section 2:

1. **Label leakage** — a feature built from the target itself.
2. **Future leakage** — a feature that wasn't knowable at the March decision point.
3. **Downstream-system leakage** — using last week's rule output as a model input instead of as a baseline to beat.

A fourth check closes the loop: an injection test that plants a deliberate leak, to prove this audit process actually catches one when it exists — an assert that never fires doesn't prove much on its own.

**1) Label leakage — passed.** `FEATURES` contains no direct copy of `decline_label` or `gsc_clicks_apr` (the next-month value the label is built from). One honest caveat worth naming rather than hiding: `gsc_clicks` and `ctr` are close cousins of the label — both are derived from March clicks, and the label is a ratio of next-month clicks to March clicks. Week 5's sensitivity check quantified how much this matters: removing both dropped Random Forest Precision@50 from 0.572 to 0.520 — a real but moderate share of the model's lift. They're kept in because they're legitimately known before the decision point (the rule itself uses `ctr` the same way) — this is a disclosed limitation, not a hidden leak.

**2) Future leakage — passed.** Every column in `X_full` (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`, `ctr`, `engagement_rate`, `peer_avg_ctr`, plus the `pos_*` position-bucket dummies) traces back to March-only aggregates. `unexpected_cols` came back empty, and `gsc_clicks_apr` is confirmed absent from `X_full` — no next-month data is present anywhere in the feature set.

**3) Downstream-system leakage — passed.** The Week 4 rule's own output (`action`, `reason_code`, `rule_score`) does not appear in `FEATURES` — `leaked_rule_cols` came back as an empty set. The rule is used only as the baseline to beat (Week 4/5's comparisons), never as something the model learns from.

**4) Injection test — confirms the audit works.** Planting a deliberate leak (`leaked_label_copy`, a direct copy of `decline_label`, added straight into `X_full`) and re-running the same GroupKFold evaluation pushed Precision@50 to **1.000**, against the real model's ~0.57–0.59 from Week 5. That gap is the point: it proves this audit process would actually catch a real leak if checks 1–3 had missed one, rather than just running asserts that happen to pass on a clean feature set by luck.

In [8]:
# 1) Label leakage check
print("FEATURES going into the model:", FEATURES)
assert 'decline_label' not in FEATURES, "label itself leaked into features"
assert 'gsc_clicks_apr' not in FEATURES, "next-month clicks (used to build the label) leaked into features"

FEATURES going into the model: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']


In [9]:
# 2) Future leakage check
march_only_source_cols = {'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
                           'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr'}
position_dummy_cols = {c for c in X_full.columns if c.startswith('pos_')}
unexpected_cols = set(X_full.columns) - march_only_source_cols - position_dummy_cols

print("Columns in X_full:", list(X_full.columns))
print("Unexpected columns (should be empty):", unexpected_cols)
assert not unexpected_cols, "X_full contains a column that isn't traceable to March-only data"
assert 'gsc_clicks_apr' not in X_full.columns

Columns in X_full: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr', 'pos_11-20', 'pos_21+', 'pos_4-10']
Unexpected columns (should be empty): set()


In [10]:
# 3) Downstream-system leakage check
rule_output_cols = {'action', 'reason_code', 'rule_score'}
leaked_rule_cols = rule_output_cols & set(FEATURES)
print("Rule-output columns found in FEATURES (should be empty):", leaked_rule_cols)
assert not leaked_rule_cols, "the rule's own output leaked into the model's input features"

Rule-output columns found in FEATURES (should be empty): set()


In [11]:
# 4) Injection test
X_leak = X_full.copy()
X_leak['leaked_label_copy'] = y  # deliberately planted leak

leak_p50 = []
for train, test in gkf.split(X_leak, y, groups):
    rf_leak = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)
    rf_leak.fit(X_leak.iloc[train], y[train])
    leak_preds = rf_leak.predict_proba(X_leak.iloc[test])[:, 1]
    test_ids = labelable['content_hash_id'].values[test]
    leak_p50.append(precision_at_k(test_ids, leak_preds, y[test], 50))

print(f"Precision@50 with a deliberately planted label-copy feature: {np.mean(leak_p50):.3f}")
print("(Compare to the real model's ~0.57–0.59 from Week 5 — a near-1.0 score here confirms")

Precision@50 with a deliberately planted label-copy feature: 1.000
(Compare to the real model's ~0.57–0.59 from Week 5 — a near-1.0 score here confirms


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (Section 2):** "...the true generalization performance is closer to ~0.68–0.70."

**Why it overreaches:** This states a fact about the model's real-world performance, but what I actually have is two point estimates — one grouped-split run and one time-aware run, each computed once, with no repeated folds or different random seeds to check how much that number would move on its own. The two estimates agreeing is a useful signal, not proof of a single true value.

**Rewritten (safe language):** Under two different honest-split methods, measured Precision@10 clustered in the 0.68–0.70 range, both well below the random split's 0.90. That's a directional signal that the random-split number overstated real-world performance — enough to use for decision-support (e.g., not shipping on the 0.90 figure) — but each estimate comes from a single run, so 0.68–0.70 should be read as a rough band, not a precise measurement.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.